In [7]:
import os
import sys
import json

from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, MapType
from delta import configure_spark_with_delta_pip
import pyarrow
from open_sky_pipeline.Connect import get_spark
from opensky_api import OpenSkyApi, TokenManager


In [ ]:
# print(sys.version)
# print(f"PyArrow włączony: {arrow_enabled}")
# print(spark.version)
# arrow_enabled = spark.conf.get("spark.sql.execution.arrow.pyspark.enabled", "false")
# print(spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

3.14.6 (main, Jul 23 2026, 14:44:54) [MSC v.1944 64 bit (AMD64)]
4.1.2


In [8]:

print(f"[INFO] HADOOP_HOME set to: {os.environ['HADOOP_HOME']}")
print(f"[INFO] PYSPARK_PYTHON set to: {os.environ['PYSPARK_PYTHON']}")
print(f"[INFO] PYSPARK_DRIVER_PYTHON set to: {os.environ['PYSPARK_DRIVER_PYTHON']}")
print(f"Zainstalowana wersja PyArrow: {pyarrow.__version__}")
print(os.environ.get("JAVA_HOME"))



[INFO] HADOOP_HOME set to: C:\hadoop
[INFO] PYSPARK_PYTHON set to: c:\Users\Si3ma\Desktop\spark_simulation\open_sky_pipeline\.venv\Scripts\python.exe
[INFO] PYSPARK_DRIVER_PYTHON set to: c:\Users\Si3ma\Desktop\spark_simulation\open_sky_pipeline\.venv\Scripts\python.exe
Zainstalowana wersja PyArrow: 25.0.1
C:\Program Files\Java\jdk-17.0.2


In [9]:
spark=get_spark()

In [10]:
with OpenSkyApi(token_manager=TokenManager.from_json_file("credentials.json")) as api:
    states = api.get_states()
states_df = states

In [11]:
aircraft_df=[]
lista={}
jdbc_url = "jdbc:postgresql://localhost:5432/OpenSky"
target_table = 'opensky_aircraft_bronze'
schema = StructType([
StructField("icao24", StringType(), True),
StructField("callsign",  StringType(), True),
StructField("origin_country",  StringType(), True),
StructField("time_position",  StringType(), True),
StructField("last_contact",  StringType(), True),
StructField("longitude",  StringType(), True),
StructField("latitude",  StringType(), True),
StructField("geo_altitude",  StringType(), True),
StructField("on_ground",  StringType(), True),
StructField("velocity",  StringType(), True),
StructField("true_track",  StringType(), True),
StructField("vertical_rate",  StringType(), True),
StructField("sensors",  StringType(), True),
StructField("baro_altitude",  StringType(), True),
StructField("squawk",  StringType(), True),
StructField("spi",  StringType(), True),
StructField("position_source",  StringType(), True),
StructField("category",  StringType(), True)
])


In [12]:

for state in states.states:
    aircraft_df.extend(
    [(
    state.icao24,
    state.callsign,
    state.origin_country,
    state.time_position,
    state.last_contact,
    state.longitude,
    state.latitude,
    state.geo_altitude,
    state.on_ground,
    state.velocity,
    state.true_track,
    state.vertical_rate,
    state.sensors,
    state.baro_altitude,
    state.squawk,
    state.spi,
    state.position_source,
    state.category
    )])

In [27]:

# for state in states.states:
#     lista[state.icao24] = {}
#     lista[state.icao24]['callsign'] = state.callsign
#     lista[state.icao24]['origin_country'] = state.origin_country
#     lista[state.icao24]['time_position'] = state.time_position
#     lista[state.icao24]['last_contact'] = state.last_contact
#     lista[state.icao24]['longitude'] = state.longitude
#     lista[state.icao24]['latitude'] = state.latitude
#     lista[state.icao24]['geo_altitude'] = state.geo_altitude
#     lista[state.icao24]['on_ground'] = state.on_ground
#     lista[state.icao24]['velocity'] = state.velocity
#     lista[state.icao24]['true_track'] = state.true_track
#     lista[state.icao24]['vertical_rate'] = state.vertical_rate
#     lista[state.icao24]['sensors'] = state.sensors
#     lista[state.icao24]['baro_altitude'] = state.baro_altitude
#     lista[state.icao24]['squawk'] = state.squawk
#     lista[state.icao24]['spi'] = state.spi
#     lista[state.icao24]['position_source'] = state.position_source
#     lista[state.icao24]['category'] = state.category
    
# # can be only one loop used !!!
# for aircraft in lista:
#     aircraft_df.extend(
#     [(
#         aircraft,
#         lista[aircraft]['callsign'],
#         lista[aircraft]['origin_country'], 
#         lista[aircraft]['time_position'],
#         lista[aircraft]['last_contact'],
#         lista[aircraft]['longitude'],
#         lista[aircraft]['latitude'],
#         lista[aircraft]['geo_altitude'],
#         lista[aircraft]['on_ground'],
#         lista[aircraft]['velocity'],
#         lista[aircraft]['true_track'],
#         lista[aircraft]['vertical_rate'],
#         lista[aircraft]['sensors'],
#         lista[aircraft]['baro_altitude'],
#         lista[aircraft]['squawk'],
#         lista[aircraft]['spi'],
#         lista[aircraft]['position_source'],
#         lista[aircraft]['category']
#         )])

In [13]:
aircraft_spark_df = spark.createDataFrame(aircraft_df, schema=schema)
# aircraft_spark_df.show()
# aircraft_spark_df.explain(True)

In [14]:
aircraft_spark_df.printSchema()

root
 |-- icao24: string (nullable = true)
 |-- callsign: string (nullable = true)
 |-- origin_country: string (nullable = true)
 |-- time_position: string (nullable = true)
 |-- last_contact: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- geo_altitude: string (nullable = true)
 |-- on_ground: string (nullable = true)
 |-- velocity: string (nullable = true)
 |-- true_track: string (nullable = true)
 |-- vertical_rate: string (nullable = true)
 |-- sensors: string (nullable = true)
 |-- baro_altitude: string (nullable = true)
 |-- squawk: string (nullable = true)
 |-- spi: string (nullable = true)
 |-- position_source: string (nullable = true)
 |-- category: string (nullable = true)



In [15]:

aircraft_spark_df = aircraft_spark_df.withColumn("ingestion_timestamp", current_timestamp())

In [20]:
output_path = f"s3a://{minio_bucket}/{target_table}"

In [ ]:
try:
    aircraft_spark_df.write \
        .format("delta") \
        .mode("append") \
        .save("s3a://bronze/aircraft")
except Exception as e:
    print(e)

In [11]:
with open("db.json", "r") as f:
    connection_properties = json.load(f)

aircraft_spark_df.write \
.mode("append") \
.jdbc(url=jdbc_url, table=target_table, properties=connection_properties)